# NB06: Synthesis & Product Recommendations

**Capstone: Connecting the Dots Across All Analyses**

This notebook synthesizes findings from NB02-NB05 into actionable product recommendations for SmarterDx.

## The Power of Synthesis

While individual analyses reveal important signals, **the real value comes from connecting dots across multiple datasets**.

We've analyzed:
- **NB02**: Acquisition funnel (landing_page → signup → product_view → add_to_cart → purchase)
- **NB03**: A/B testing (comparing AI model versions)
- **NB04**: Cohort analysis (users by signup date)
- **NB05**: Retention & churn (continued adoption of product)

Now we translate these insights into a prioritized roadmap that balances impact and effort.

### Mapping Concepts
| Generic Concept | Data Structure |
|---|---|
| **Funnel Stages** | landing_page, signup, product_view, add_to_cart, purchase |
| **A/B Testing** | control/treatment groups, converted (0/1) |
| **Cohorts** | Users grouped by signup_cohort (W01-W12) |
| **Retention** | Activity events by weeks_since_signup |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up paths
data_dir = Path('../data/inputs')
output_dir = Path('../data/outputs/nb06')
output_dir.mkdir(parents=True, exist_ok=True)

# Define color palette
colors = ['#2E86AB', '#F18F01', '#2CA58D', '#E15554']
plt.style.use('default')
sns.set_palette(colors)

print(f'Output directory: {output_dir}')
print(f'Files in data_dir: {list(data_dir.glob("*.csv"))}')

# Load datasets
funnel_df = pd.read_csv(data_dir / 'funnel_clean.csv')
ab_df = pd.read_csv(data_dir / 'ab_clean.csv')
activity_df = pd.read_csv(data_dir / 'activity_clean.csv')

print(f'Loaded funnel_clean.csv: {funnel_df.shape}')
print(f'Loaded ab_clean.csv: {ab_df.shape}')
print(f'Loaded activity_clean.csv: {activity_df.shape}')

In [ ]:
# Quick data inspection
print('Funnel columns:', funnel_df.columns.tolist())
print('Funnel head:')
print(funnel_df.head())
print('\nAB columns:', ab_df.columns.tolist())
print('AB head:')
print(ab_df.head())
print('\nActivity columns:', activity_df.columns.tolist())
print('Activity head:')
print(activity_df.head())

## Executive Summary Dashboard

A 2x2 grid capturing the four key metrics that drive product success.

In [ ]:
# Prepare data for the 4-panel dashboard\n\n# Panel 1: Funnel Conversion\nfunnel_stages = ['landing_page', 'signup', 'product_view', 'add_to_cart', 'purchase']\nfunnel_conversion = []\nfor stage in funnel_stages:\n    users_at_stage = (funnel_df['stage'] == stage).sum()\n    funnel_conversion.append(users_at_stage)\n\nprint('Funnel conversion (users at each stage):')\nfor stage, count in zip(funnel_stages, funnel_conversion):\n    print(f'{stage}: {count}')\n\n# Panel 2: A/B Test Conversion\nab_conversion = ab_df.groupby('group')['converted'].mean() * 100\nprint(f'\\nA/B conversion rates:\\n{ab_conversion}')\n\n# Panel 3: Cohort Retention Heatmap\nactivity_df['signup_date_dt'] = pd.to_datetime(activity_df['signup_date'])\nactivity_df['activity_date'] = pd.to_datetime(activity_df['activity_date'])\nactivity_df['days_since_signup'] = (activity_df['activity_date'] - activity_df['signup_date_dt']).dt.days\nactivity_df['weeks_since_signup'] = activity_df['days_since_signup'] // 7\n\n# Pivot for heatmap: signup_cohort vs weeks_since_signup\ncohort_retention = activity_df.groupby(['signup_cohort', 'weeks_since_signup']).size().unstack(fill_value=0)\ncohort_retention_pct = cohort_retention.div(cohort_retention.iloc[:, 0], axis=0) * 100\ncohort_retention_pct = cohort_retention_pct.iloc[:, :8]  # Show first 8 weeks\n\nprint(f'\\nCohort retention heatmap shape: {cohort_retention_pct.shape}')\n\n# Panel 4: Overall Retention Curve\nweekly_retention = activity_df.groupby('weeks_since_signup').size()\ntotal_users = activity_df['user_id'].nunique()\nretention_pct = (weekly_retention / total_users) * 100\n\nprint(f'\\nOverall retention curve (first 10 weeks):')\nprint(retention_pct.head(10))

In [ ]:
# Create the 2x2 Executive Dashboard\nfig, axes = plt.subplots(2, 2, figsize=(14, 10))\nfig.suptitle('Executive Summary Dashboard: Key Product Metrics', fontsize=16, fontweight='bold', y=0.995)\n\n# Panel 1: Funnel Conversion Bar Chart (Top-Left)\nax1 = axes[0, 0]\nstage_labels = ['Landing Page', 'Signup', 'Product View', 'Add to Cart', 'Purchase']\ncolors_funnel = [colors[0], colors[1], colors[2], colors[3], colors[0]]\nax1.bar(stage_labels, funnel_conversion, color=colors_funnel, alpha=0.8, edgecolor='black', linewidth=1.5)\nax1.set_title('Acquisition Funnel', fontsize=12, fontweight='bold')\nax1.set_ylabel('Users Reached', fontsize=10)\nax1.set_xlabel('Stage', fontsize=10)\nfor i, v in enumerate(funnel_conversion):\n    ax1.text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold')\nax1.set_ylim(0, max(funnel_conversion) * 1.15)\nax1.grid(axis='y', alpha=0.3)\nax1.tick_params(axis='x', rotation=45)\n\n# Panel 2: A/B Test Conversion (Top-Right)\nax2 = axes[0, 1]\ngroups = ab_conversion.index.tolist()\nconversion_rates = ab_conversion.values.tolist()\nax2.bar(groups, conversion_rates, color=[colors[0], colors[1]], alpha=0.8, edgecolor='black', linewidth=1.5)\nax2.set_title('A/B Test: Model Version Comparison', fontsize=12, fontweight='bold')\nax2.set_ylabel('Conversion Rate (%)', fontsize=10)\nax2.set_xlabel('Model Group', fontsize=10)\nfor i, v in enumerate(conversion_rates):\n    ax2.text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')\nax2.set_ylim(0, max(conversion_rates) * 1.15)\nax2.grid(axis='y', alpha=0.3)\n\n# Panel 3: Cohort Retention Heatmap (Bottom-Left)\nax3 = axes[1, 0]\nsns.heatmap(cohort_retention_pct, cmap='RdYlGn', annot=True, fmt='.0f', cbar_kws={'label': '% Retained'},\n            ax=ax3, vmin=0, vmax=100, linewidths=0.5)\nax3.set_title('Cohort Retention: Weeks Since Signup', fontsize=12, fontweight='bold')\nax3.set_xlabel('Weeks Since Signup', fontsize=10)\nax3.set_ylabel('Signup Cohort', fontsize=10)\n\n# Panel 4: Overall Retention Curve (Bottom-Right)\nax4 = axes[1, 1]\nweeks = retention_pct.index[:13]\nretention_vals = retention_pct.values[:13]\nax4.plot(weeks, retention_vals, marker='o', linewidth=2.5, markersize=6, color=colors[2])\nax4.fill_between(weeks, retention_vals, alpha=0.3, color=colors[2])\nax4.set_title('Overall User Retention Curve', fontsize=12, fontweight='bold')\nax4.set_xlabel('Weeks Since Signup', fontsize=10)\nax4.set_ylabel('% Active Users', fontsize=10)\nax4.grid(True, alpha=0.3)\nax4.set_ylim(0, 105)\n\nplt.tight_layout()\nplt.savefig(output_dir / 'nb06_executive_dashboard.png', dpi=300, bbox_inches='tight')\nprint(f'Saved executive dashboard to {output_dir / \"nb06_executive_dashboard.png\"}')\nplt.show()

## Product Story: Situation → Complication → Resolution

### **Situation**
SmarterDx has built an innovative AI system for clinical decision support. Early hospitals show strong initial adoption, but we're seeing variation in how long they remain active.

### **Complication**
While our A/B test shows Model V2 outperforms V1 (higher conversion), conversion alone doesn't guarantee long-term success. Retention curves flatten after week 6-8 for some cohorts. Key questions:
- Why do some hospitals disengage after go-live?
- What drives multi-month adoption vs. churn?
- How do model improvements translate to sustainable product engagement?

### **Resolution**
By combining funnel, A/B, and retention insights, we've identified a 3-pillar roadmap:
1. **Improve onboarding quality** (Acquisition)
2. **Optimize model performance** (Monetization via A/B testing)
3. **Build retention mechanisms** (long-term value)

This positions SmarterDx to move from single-transaction wins (go-live) to long-term partnerships (multi-year adoption).

## Prioritized Recommendations: Impact vs Effort Matrix

In [ ]:
# Define 6-7 product recommendations with impact and effort scores
recommendations = [
    {'name': 'Expand Model V2\nRollout', 'impact': 8.5, 'effort': 3, 'quad': 'Quick Wins'},
    {'name': 'Automated Chart\nReview Dashboards', 'impact': 8, 'effort': 6, 'quad': 'Strategic'},
    {'name': 'In-App Onboarding\nTutorials', 'impact': 7, 'effort': 4, 'quad': 'Quick Wins'},
    {'name': 'Weekly Digest\nNotifications', 'impact': 6, 'effort': 2, 'quad': 'Quick Wins'},
    {'name': 'Custom Model\nTraining per Hospital', 'impact': 8.5, 'effort': 8.5, 'quad': 'Strategic'},
    {'name': 'Integration with EHR\nSystems', 'impact': 9, 'effort': 9, 'quad': 'Avoid'},
    {'name': 'Competitive\nBenchmarking Portal', 'impact': 5, 'effort': 7, 'quad': 'Nice-to-Have'},
]

rec_df = pd.DataFrame(recommendations)
print(rec_df)

In [ ]:
# Create impact vs effort scatter plot
fig, ax = plt.subplots(figsize=(12, 8))

# Define colors by quadrant
quad_colors = {'Quick Wins': '#2CA58D', 'Strategic': '#F18F01', 'Nice-to-Have': '#2E86AB', 'Avoid': '#E15554'}

for idx, row in rec_df.iterrows():
    ax.scatter(row['effort'], row['impact'], s=400, alpha=0.7,
               color=quad_colors[row['quad']], edgecolors='black', linewidth=2)
    ax.annotate(row['name'], (row['effort'], row['impact']),
                fontsize=9, fontweight='bold', ha='center', va='center')

# Add quadrant lines
ax.axhline(y=6.5, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)
ax.axvline(x=5.5, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)

# Add quadrant labels
ax.text(2.5, 8.5, 'QUICK WINS', fontsize=11, fontweight='bold', alpha=0.5, ha='center')
ax.text(8, 8.5, 'STRATEGIC', fontsize=11, fontweight='bold', alpha=0.5, ha='center')
ax.text(2.5, 3.5, 'NICE-TO-HAVE', fontsize=11, fontweight='bold', alpha=0.5, ha='center')
ax.text(8, 3.5, 'AVOID', fontsize=11, fontweight='bold', alpha=0.5, ha='center')

ax.set_xlabel('Effort (1=easy, 10=hard)', fontsize=12, fontweight='bold')
ax.set_ylabel('Impact (1=low, 10=high)', fontsize=12, fontweight='bold')
ax.set_title('Product Roadmap: Impact vs Effort Matrix', fontsize=14, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(3, 10)
ax.grid(True, alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, edgecolor='black', label=quad)
                   for quad, color in quad_colors.items()]
ax.legend(handles=legend_elements, loc='lower left', fontsize=10, title='Quadrant')

plt.tight_layout()
plt.savefig(output_dir / 'nb06_impact_effort_matrix.png', dpi=300, bbox_inches='tight')
print(f'Saved impact-effort matrix to {output_dir / "nb06_impact_effort_matrix.png"}')
plt.show()

## North Star Metric & Metrics Tree

The **North Star Metric** for SmarterDx: **Incremental Diagnoses Captured**

This captures the fundamental value: hospitals use our AI to catch diagnoses they would have otherwise missed.

In [ ]:
# Create a metrics tree visualization
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Define tree structure
# Level 1: North Star (top center)
north_star = FancyBboxPatch((3.5, 8.5), 3, 1, boxstyle="round,pad=0.1",
                             edgecolor='black', facecolor='#F18F01', linewidth=2.5, alpha=0.8)
ax.add_patch(north_star)
ax.text(5, 9, 'North Star: Incremental\nDiagnoses Captured', ha='center', va='center',
        fontsize=11, fontweight='bold', color='white')

# Level 2: Three pillars
pillar_y = 6
pillars = [
    {'x': 1.5, 'name': 'Acquisition', 'color': '#2E86AB'},
    {'x': 5, 'name': 'Monetization', 'color': '#2CA58D'},
    {'x': 8.5, 'name': 'Retention', 'color': '#E15554'},
]

for pillar in pillars:
    # Draw box
    box = FancyBboxPatch((pillar['x']-1, pillar_y-0.4), 2, 0.8, boxstyle="round,pad=0.05",
                          edgecolor='black', facecolor=pillar['color'], linewidth=2, alpha=0.8)
    ax.add_patch(box)
    ax.text(pillar['x'], pillar_y, pillar['name'], ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

    # Draw line from north star to pillar
    ax.plot([5, pillar['x']], [8.5, pillar_y+0.4], 'k-', linewidth=2)

# Level 3: Sub-metrics
sub_metrics = [
    {'x': 1.5, 'y': 4.5, 'text': 'Funnel\nConversion', 'pillar': 0},
    {'x': 1.5, 'y': 3, 'text': 'Pilot\nCompletion', 'pillar': 0},
    {'x': 5, 'y': 4.5, 'text': 'Model\nAccuracy', 'pillar': 1},
    {'x': 5, 'y': 3, 'text': 'A/B Test\nWinners', 'pillar': 1},
    {'x': 8.5, 'y': 4.5, 'text': 'User\nRetention', 'pillar': 2},
    {'x': 8.5, 'y': 3, 'text': 'NPS & Churn', 'pillar': 2},
]

for metric in sub_metrics:
    pillar = pillars[metric['pillar']]
    # Draw box
    sub_box = FancyBboxPatch((metric['x']-0.7, metric['y']-0.35), 1.4, 0.7, boxstyle="round,pad=0.03",
                              edgecolor='gray', facecolor='lightgray', linewidth=1.5, alpha=0.6)
    ax.add_patch(sub_box)
    ax.text(metric['x'], metric['y'], metric['text'], ha='center', va='center',
            fontsize=8, fontweight='bold')

    # Draw line from pillar to sub-metric
    ax.plot([pillar['x'], metric['x']], [pillar_y-0.4, metric['y']+0.35], 'gray', linewidth=1.5)

ax.set_title('Metrics Tree: From North Star to Actionable Metrics', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(output_dir / 'nb06_metrics_tree.png', dpi=300, bbox_inches='tight')
print(f'Saved metrics tree to {output_dir / "nb06_metrics_tree.png"}')
plt.show()

## Guardrail Strategy: The Safety Net Behind Every Recommendation

---

### Why This Section Exists

Every recommendation in the impact-effort matrix above comes with risk. A good product analyst doesn't just say *"do this"* — they say *"do this, measure these outcomes, and stop if these guardrails break."*

---

### Guardrail Matrix for Our Recommendations

| Recommendation | Primary Metric | Guardrail Metrics | Rollback Trigger |
|---|---|---|---|
| Fix mobile funnel drop-off | Mobile signup rate | Desktop signup rate (shouldn't drop), page load time | Mobile signup drops below pre-change baseline |
| Optimize checkout flow | Purchase conversion | Average order value, return rate | AOV drops >10% or returns increase >2σ |
| Run targeted onboarding for casual users | Week-1 retention for casual cohort | Email unsubscribe rate, support ticket volume | Unsubscribe rate exceeds 5% |
| Launch win-back campaign for churned users | Reactivation rate | Cost per reactivation, spam complaint rate | CPA exceeds 3× the value of a reactivated user |
| A/B test new feature for engagement | Feature adoption rate | Session error rate, time-on-task | Error rate increases or task completion drops |

---

### North Star Health Check

The North Star Metric (Purchase Conversion Rate, or in SmarterDx terms, Incremental Diagnoses Captured) should be reviewed weekly with this checklist:

1. **Is the North Star moving in the right direction?** (Trend line)
2. **Are any guardrails breaching thresholds?** (Automated alerts)
3. **Are the supporting metrics (acquisition, monetization, retention) all contributing?** (Metrics tree health)
4. **Is there a segment where the North Star is declining even if the overall number looks fine?** (Segment-level cuts)

---

### Interview Tip: The Guardrail Answer Pattern

When an interviewer asks *"How would you measure the success of this product?"*, a strong answer follows this pattern:

> "The primary metric I'd track is [North Star or experiment metric]. But I'd also set guardrails on [2-3 things that shouldn't get worse], with automated alerts if any guardrail degrades by [threshold]. If we see [specific rollback trigger], we'd pause the rollout and investigate before continuing."

This shows you think about unintended consequences — which is especially important in healthcare where a false positive from an AI system can directly affect patient care.

## SmarterDx Clinical AI Bridge: From Generic to Domain-Specific

How do our findings translate to SmarterDx's business model?

### Mapping Concepts: Generic E-Commerce to Product Analytics

| Generic Concept | Data Column/Value | Real-World Meaning |
|---|---|---|
| **Funnel Stages** | `stage` column: landing_page, signup, product_view, add_to_cart, purchase | User progression through acquisition journey |
| **A/B Testing** | `group` (control/treatment), `converted` (0/1) | Comparing conversion effectiveness of different approaches |
| **Cohorts** | `signup_cohort` (W01-W12) | Grouping users by signup week to track retention patterns |
| **Retention** | `weeks_since_signup`, activity events | Measuring sustained engagement over time |
| **North Star** | Composite: Conversion Rate × Retention Rate | Long-term user value (acquisition + stickiness) |

### Why This Matters

The power of synthesis comes from understanding how these metrics interconnect:
1. **Funnel alone** tells us how many users reach each stage, but not if they stay
2. **A/B tests alone** tell us which approach converts better, but not which drives long-term value
3. **Retention alone** tells us who stays, but not how we acquired them or what made them convert
4. **Together**: We can optimize for users who both convert AND retain—the most valuable segment

## Interview Cheat Sheet: 6-Step Framework for Case Study Answers

When asked "Walk us through your analysis," use this structure:

### 1. **Clarify the Problem** (30 seconds)
   - State the business question: "How do we improve long-term user adoption of our product?"
   - Identify key unknowns: "We needed to understand both acquisition funnels AND retention patterns."

### 2. **Outline Your Approach** (30 seconds)
   - "I analyzed 3 datasets: funnel (acquisition), A/B tests (model quality), and activity (retention)."
   - "Rather than solve in isolation, I connected the dots across all three."

### 3. **Share Key Findings** (2 minutes)
   - **Acquisition**: Funnel shows clear drop-off; stages progress from landing_page → signup → product_view → add_to_cart → purchase
   - **Optimization**: A/B test shows one model group significantly outperforms the other in conversion
   - **Retention**: Activity patterns show variation across cohorts; some retain 80% through week 8, others drop to 20%

### 4. **Synthesize Insights** (1 minute)
   - "The pattern: strong acquisition + good model quality alone ≠ sticky product"
   - "Users churn because they lack ongoing engagement signals and value demonstrations"

### 5. **Recommend Actions** (1 minute)
   - **Quick wins**: Notification digests (high impact, low effort)
   - **Strategic bets**: Custom model optimization per user segment (high impact, high effort)
   - **Avoid**: Full system integration (too risky, too expensive in year 1)

### 6. **Discuss Trade-offs & Next Steps** (30 seconds)
   - "Quick wins fund strategic bets. We recommend deploying winning model variant + notifications in Q1, custom models in Q2."
   - "Success metric: increase 12-week retention from current baseline by 20+ percentage points by end of year."

## Final Takeaways

### For Product Teams
- **Synthesis > Individual Analysis**: Funnel + A/B + Retention insights together drive strategy
- **Impact-Effort Tradeoffs Matter**: Quick wins fund strategic bets; know the difference
- **North Star Alignment**: Metrics tree ensures everyone optimizes the same outcome

### For SmarterDx
- **Expand Model V2**: A/B test shows clear winner—no reason to wait
- **Build Retention Infrastructure**: Email digests, dashboards, and integrations are cheaper than re-acquisition
- **Measure Clinical Outcomes**: "Incremental diagnoses captured" beats "usage hours"—that's what customers care about

### For Your Career
- **Tell the Story**: Don't just show graphs; show how pieces fit together
- **Prioritize Ruthlessly**: Not everything matters equally; explicit scoring (impact/effort) beats gut feel
- **Bridge Domains**: Translate analytics into business decisions; that's where the value lives

## Reference: Earlier Notebooks

- **NB02**: Acquisition funnel analysis (landing_page → signup → product_view → add_to_cart → purchase)
- **NB03**: A/B testing framework (comparing model variants)
- **NB04**: Cohort analysis (users by signup date and signup_cohort)
- **NB05**: Retention & churn modeling (weeks since signup)
- **NB06** (this notebook): Synthesis & prioritized recommendations